In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [17]:
!pip install entsoe-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.3 MB/s eta 0:00:00


In [18]:
import os
import pandas as pd
import pulp
from entsoe import EntsoePandasClient

# 1. Initialize ENTSO-E Client (Requires free API key from transparency.entsoe.eu)
# You can set it as an environment variable or paste it directly here
api_key = os.environ.get("ENTSOE_API_KEY", "YOUR_API_KEY_HERE")
client = EntsoePandasClient(api_key=api_key)

# 2. Define time range for a high negative-price day in Germany (e.g., a sunny spring/summer weekend)
start = pd.Timestamp('2026-05-14 00:00:00', tz='Europe/Brussels')
end = pd.Timestamp('2026-05-14 23:00:00', tz='Europe/Brussels')
country_code = 'DE_LU'  # Germany-Luxembourg bidding zone

try:
    # Fetch real Day-Ahead spot prices from ENTSO-E API
    prices_series = client.query_day_ahead_prices(country_code, start=start, end=end)
    prices = prices_series.values[:24]  .tolist()
    print("Successfully fetched real EPEX Spot prices from ENTSO-E!")
except Exception as e:
    print(f"API fetch failed ({e}), falling back to synthetic high-volatility price profile.")
    prices = [40, 30, 20, 10, 2, -15, -45, -20, 5, 30, 70, 95,
              110, 85, 40, 20, 15, 60, 120, 150, 130, 95, 70, 50]

hours = range(24)

# Dynamic grid export limit (MW) simulating grid congestion during peak solar hours
grid_export_limit = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.3, 0.1, 0.4, 1.0, 1.0, 1.0,
                     0.0, 0.0, 0.2, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

# Battery specifications
max_power = 1.0          # Max power (MW)
max_capacity = 2.0       # Max energy (MWh)
efficiency = 0.92        # Round-trip efficiency
degradation_cost = 5.0   # EUR per MWh throughput

# 3. MILP Optimization Model
prob = pulp.LpProblem("BESS_Real_Market_Optimization", pulp.LpMaximize)

P_charge = {t: pulp.LpVariable(f"P_charge_{t}", lowBound=0, upBound=max_power) for t in hours}
P_discharge = {t: pulp.LpVariable(f"P_discharge_{t}", lowBound=0, upBound=max_power) for t in hours}
SoC = {t: pulp.LpVariable(f"SoC_{t}", lowBound=0, upBound=max_capacity) for t in hours}
u = {t: pulp.LpVariable(f"u_{t}", cat='Binary') for t in hours}

revenue = pulp.lpSum([prices[t] * (P_discharge[t] - P_charge[t]) for t in hours])
cost_deg = pulp.lpSum([degradation_cost * (P_charge[t] + P_discharge[t]) for t in hours])
prob += revenue - cost_deg

initial_soc = 1.0
for t in hours:
    prob += P_discharge[t] <= grid_export_limit[t]
    prob += P_charge[t] <= max_power * u[t]
    prob += P_discharge[t] <= max_power * (1 - u[t])

    if t == 0:
        prob += SoC[t] == initial_soc + (P_charge[t] * efficiency) - (P_discharge[t] / efficiency)
    else:
        prob += SoC[t] == SoC[t-1] + (P_charge[t] * efficiency) - (P_discharge[t] / efficiency)

prob.solve(pulp.PULP_CBC_CMD(msg=0))

# 4. Process results
results = []
total_revenue = 0
for t in hours:
    net_flow = P_discharge[t].varValue - P_charge[t].varValue
    hourly_rev = prices[t] * net_flow
    total_revenue += hourly_rev
    results.append({
        'Hour': t,
        'Price (€/MWh)': round(prices[t], 2),
        'Grid_Limit (MW)': grid_export_limit[t],
        'Charge (MW)': round(P_charge[t].varValue, 3),
        'Discharge (MW)': round(P_discharge[t].varValue, 3),
        'SoC (MWh)': round(SoC[t].varValue, 3),
        'Net Revenue (€)': round(hourly_rev, 2)
    })

df_res = pd.DataFrame(results)
print(df_res.to_string())
print(f"\nTotal Optimized Daily Revenue with Real Market Data: €{round(total_revenue, 2)}")

API fetch failed (401 Client Error:  for url: https://web-api.tp.entsoe.eu/api?documentType=A44&in_Domain=10Y1001A1001A82H&out_Domain=10Y1001A1001A82H&offset=0&contract_MarketAgreement.type=A01&classificationSequence_AttributeInstanceComponent.position=1&securityToken=YOUR_API_KEY_HERE&periodStart=202605122200&periodEnd=202605152100), falling back to synthetic high-volatility price profile.
    Hour  Price (€/MWh)  Grid_Limit (MW)  Charge (MW)  Discharge (MW)  SoC (MWh)  Net Revenue (€)
0      0             40              1.0        0.000            0.92      0.000            36.80
1      1             30              1.0        0.000            0.00      0.000             0.00
2      2             20              1.0        0.000            0.00      0.000             0.00
3      3             10              1.0        0.000            0.00      0.000             0.00
4      4              2              1.0        0.000            0.00      0.000             0.00
5      5          

In [19]:
import pulp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Fallback High-Volatility Market Prices (Including negative price intervals)
prices = [40, 30, 20, 10, 2, -15, -45, -20, 5, 30, 70, 95,
          110, 85, 40, 20, 15, 60, 120, 150, 130, 95, 70, 50]
hours = range(24)

# Dynamic grid export limit (MW) simulating grid congestion during peak solar hours
grid_export_limit = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.3, 0.1, 0.4, 1.0, 1.0, 1.0,
                     0.0, 0.0, 0.2, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

# Battery specifications
max_power = 1.0          # Max power (MW)
max_capacity = 2.0       # Max energy (MWh)
efficiency = 0.92        # Round-trip efficiency
degradation_cost = 5.0   # EUR per MWh throughput

# 2. MILP Optimization Model
prob = pulp.LpProblem("BESS_Real_Market_Optimization", pulp.LpMaximize)

P_charge = {t: pulp.LpVariable(f"P_charge_{t}", lowBound=0, upBound=max_power) for t in hours}
P_discharge = {t: pulp.LpVariable(f"P_discharge_{t}", lowBound=0, upBound=max_power) for t in hours}
SoC = {t: pulp.LpVariable(f"SoC_{t}", lowBound=0, upBound=max_capacity) for t in hours}
u = {t: pulp.LpVariable(f"u_{t}", cat='Binary') for t in hours}

revenue = pulp.lpSum([prices[t] * (P_discharge[t] - P_charge[t]) for t in hours])
cost_deg = pulp.lpSum([degradation_cost * (P_charge[t] + P_discharge[t]) for t in hours])
prob += revenue - cost_deg

initial_soc = 1.0
for t in hours:
    prob += P_discharge[t] <= grid_export_limit[t]
    prob += P_charge[t] <= max_power * u[t]
    prob += P_discharge[t] <= max_power * (1 - u[t])

    if t == 0:
        prob += SoC[t] == initial_soc + (P_charge[t] * efficiency) - (P_discharge[t] / efficiency)
    else:
        prob += SoC[t] == SoC[t-1] + (P_charge[t] * efficiency) - (P_discharge[t] / efficiency)

prob.solve(pulp.PULP_CBC_CMD(msg=0))

# 3. Process results
results = []
total_revenue = 0
for t in hours:
    net_flow = P_discharge[t].varValue - P_charge[t].varValue
    hourly_rev = prices[t] * net_flow
    total_revenue += hourly_rev
    results.append({
        'Hour': t,
        'Price (€/MWh)': round(prices[t], 2),
        'Grid_Limit (MW)': grid_export_limit[t],
        'Charge (MW)': round(P_charge[t].varValue, 3),
        'Discharge (MW)': round(P_discharge[t].varValue, 3),
        'SoC (MWh)': round(SoC[t].varValue, 3),
        'Net Revenue (€)': round(hourly_rev, 2)
    })

df_res = pd.DataFrame(results)
print(df_res.to_string())
print(f"\nTotal Optimized Daily Revenue: €{round(total_revenue, 2)}")

    Hour  Price (€/MWh)  Grid_Limit (MW)  Charge (MW)  Discharge (MW)  SoC (MWh)  Net Revenue (€)
0      0             40              1.0        0.000            0.92      0.000            36.80
1      1             30              1.0        0.000            0.00      0.000             0.00
2      2             20              1.0        0.000            0.00      0.000             0.00
3      3             10              1.0        0.000            0.00      0.000             0.00
4      4              2              1.0        0.000            0.00      0.000             0.00
5      5            -15              1.0        0.174            0.00      0.160             2.61
6      6            -45              0.3        1.000            0.00      1.080            45.00
7      7            -20              0.1        1.000            0.00      2.000            20.00
8      8              5              0.4        0.000            0.00      2.000             0.00
9      9            